In [10]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [11]:
import os

key = os.getenv("OPENAI_API_KEY")

# print(key)

In [ ]:
# from langchain.llms.openai import OpenAI
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate, ChatPromptTemplate

# llm = OpenAI()
chat = ChatOpenAI(model_name="gpt-4o-mini", temperature=1)

# template, prompt
template = PromptTemplate.from_template("What is the distance between {country_a} and {country_b}? Also, what is your name?")
prompt = template.format(country_a="Mexico", country_b="Thailand")

chat.predict(prompt)

'The distance between Mexico and Thailand varies depending on the specific locations you are measuring between. For example, the distance from Mexico City to Bangkok is approximately 9,300 kilometers (about 5,800 miles). \n\nAs for my name, I am an AI language model created by OpenAI, and I don\'t have a personal name like a human would. You can simply refer to me as "Assistant." How can I help you further?'

In [ ]:
template = ChatPromptTemplate.from_messages(
    [
       ("system", "You are a geography expert. And you only reply in {language}."),
       ("ai", "Ciao, mi chiamo {name}!"),
       ("human", "What is the distance between {country_a} and {country_b}? Also, what is your name?")
   ]
)

prompt = template.format_messages(
    language="Greek",
    name="Socrates",
    country_a="Mexico",
    country_b="Thailand"
)

chat.predict_messages(prompt)

AIMessage(content='Η απόσταση μεταξύ Μεξικού και Ταϊλάνδης είναι περίπου 13.000 χιλιόμετρα, ανάλογα με την ακριβή τοποθεσία που συγκρίνετε. Το όνομά μου είναι Σωκράτης!')

In [24]:
from langchain.schema import BaseOutputParser

class CommaOutputParser(BaseOutputParser):
    def parse(self, text):
        items = text.strip().split(",") # 앞뒤 공백 제거 후 "," 기준으로 분리후 리스트로 받는다.

        return list(map(str.strip, items)) # item요소의 순회 해서 앞뒤 공백을 제거한다.

p = CommaOutputParser()
p.parse("Hello, how, are, you")

['Hello', 'how', 'are', 'you']

In [ ]:
template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a list generating machine. Everything you are asked will be answered with a comma seperated list of max {max_items} in lowercase. Do NOT reply with anything else."),
        ("human", "{question}")
    ]
)

In [32]:
chain = template | chat | CommaOutputParser()

chain.invoke({
    "max_items": 5,
    "question": "What are the pokemons?"
})

['pikachu', 'charmander', 'bulbasaur', 'squirtle', 'jigglypuff']

In [36]:
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(temperature=1, streaming=True, callbacks=[StreamingStdOutCallbackHandler()])

chef_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a world-class international chef. You create easy to follow recipes for any type of cuisine with easy to find ingredients."),
    ("human", "I want to cook {cuisine} food.")
])

chef_chain = chef_prompt | chat


In [37]:
veg_chef_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a vegetarian chef specialized on making traditional recipes vegetarian. You find alternative ingredients and explain their preparation. You don't radically modify the recipe. If there is no alternative for a food just say you don't know how to replace it."
    ),
    (
        "human",
        "{recipe}"
    )
])

veg_chain = veg_chef_prompt | chat

final_chain = {"recipe" : chef_chain} | veg_chain

final_chain.invoke({
    "cuisine" : "indian"
})

Great choice! Indian cuisine is known for its bold flavors and aromatic spices. Let's start with a classic and popular dish, Chicken Tikka Masala. Here's a simple and delicious recipe for you to try at home:

Chicken Tikka Masala:

Ingredients:
- 1 lb boneless, skinless chicken breasts, cut into bite-sized pieces
- 1 cup plain yogurt
- 2 tablespoons olive oil
- 1 onion, finely chopped
- 3 cloves garlic, minced
- 1-inch piece of ginger, minced
- 1 can (14 oz) diced tomatoes
- 2 tablespoons tomato paste
- 1 teaspoon ground cumin
- 1 teaspoon ground coriander
- 1 teaspoon paprika
- 1 teaspoon garam masala
- 1/2 teaspoon turmeric
- Salt and pepper, to taste
- Fresh cilantro, for garnish
- Cooked rice or naan, for serving

Instructions:
1. In a bowl, combine the yogurt, olive oil, half of the minced garlic, half of the minced ginger, ground cumin, ground coriander, paprika, garam masala, turmeric, salt, and pepper. Add the chicken pieces and mix well to coat. Cover and marinate in the refri

AIMessageChunk(content="As a vegetarian chef specializing in making traditional recipes vegetarian, I can offer you an alternative recipe for Chicken Tikka Masala using plant-based ingredients. Here's the modified recipe:\n\nVegetarian Tikka Masala:\n\nIngredients:\n- 1 lb firm tofu, pressed and cut into bite-sized cubes (to mimic the texture of chicken)\n- 1 cup plain vegan yogurt\n- 2 tablespoons olive oil\n- 1 onion, finely chopped\n- 3 cloves garlic, minced\n- 1-inch piece of ginger, minced\n- 1 can (14 oz) diced tomatoes\n- 2 tablespoons tomato paste\n- 1 teaspoon ground cumin\n- 1 teaspoon ground coriander\n- 1 teaspoon paprika\n- 1 teaspoon garam masala\n- 1/2 teaspoon turmeric\n- Salt and pepper, to taste\n- Fresh cilantro, for garnish\n- Cooked rice or naan, for serving\n\nInstructions:\n1. In a bowl, combine the vegan yogurt, olive oil, half of the minced garlic, half of the minced ginger, ground cumin, ground coriander, paprika, garam masala, turmeric, salt, and pepper. Add 